# 4.0 - CNN (DenseNet201, transfer learning)

DenseNet201 pre-treinada no ImageNet, ajustada para real vs. ataque no NUAA.
A entrada sao as mesmas imagens 160x160 RGB geradas pelo MTCNN e usadas no
LBP e no HOG (sem resize).

A variavel estudada e **quanto da rede e destravado** no fine-tuning. Troque
`EXPERIMENT` na primeira celula de configuracao e rode o notebook; cada
valor e um experimento:

| EXPERIMENT | O que treina |
|---|---|
| `cnn_01_head` | so a camada final nova |
| `cnn_02_partial` | `denseblock4` + `norm5` + camada final |
| `cnn_03_full` | a rede inteira |

Cada experimento e repetido com 3 seeds. O modelo e avaliado primeiro na
validacao; o teste so roda na secao final, uma vez por experimento.

**Referencias das decisoes**
- DenseNet201, Adam, entropia cruzada binaria, dropout 0,5, augmentation
  (flip, rotacao, brilho) e early stopping pela loss de validacao: CMES,
  *Optimizing CNN Architectures for Face Liveness Detection*.
- Niveis de congelamento: Yosinski et al. (2014), *How transferable are
  features in deep neural networks?*.
- Decisoes nossas (nao vem de artigo): bloco destravado no parcial,
  magnitudes do augmentation, learning rates das camadas destravadas,
  paciencia, maximo de epocas, batch e seeds.

In [ ]:
# Clona o repositorio e instala as dependencias.
# facenet-pytorch precisa de --no-deps (ver README). Esse notebook nao roda
# MTCNN, mas mantemos o mesmo setup dos outros notebooks.

!git clone https://github.com/laianemuckler/liveness-detection.git
%cd liveness-detection

!pip install -r requirements.txt --quiet
!pip install facenet-pytorch==2.6.0 --no-deps --quiet

In [ ]:
# Monta o Drive

from google.colab import drive
drive.mount('/content/drive')

## Configuracao

`EXPERIMENT` e a unica coisa que muda entre as execucoes. O resto fica fixo
para que os 3 experimentos sejam comparaveis.

In [ ]:
# Experimento a rodar: 'cnn_01_head', 'cnn_02_partial' ou 'cnn_03_full'
EXPERIMENT = 'cnn_01_head'

# Hiperparametros (iguais nos 3 experimentos)
SEEDS = [42, 43, 44]
BATCH_SIZE = 32
MAX_EPOCHS = 30
PATIENCE = 5            # early stopping pela loss de validacao
LR_HEAD = 1e-3          # camada final nova (valor do CMES)
LR_BACKBONE = 1e-4      # camadas destravadas que vem do ImageNet
DROPOUT = 0.5           # CMES
ROTATION_DEGREES = 10   # augmentation (tipos do CMES, valores nossos)
BRIGHTNESS = 0.2

# Textos gravados no experiment_log.csv (iguais nas linhas de validacao e teste)
METODO = 'DenseNet201'
FEATURE_CONFIG = f"{EXPERIMENT}, 160x160 RGB"
MODELO_CONFIG = (f"adam lr_head={LR_HEAD} lr_backbone={LR_BACKBONE}, dropout={DROPOUT}, "
                 f"batch={BATCH_SIZE}, patience={PATIENCE}")

In [ ]:
# Importa as funcoes do projeto

import os
import numpy as np
import pandas as pd
import torch

from src.config import TRAIN_DIR, VAL_DIR, TEST_DIR, DRIVE_ROOT
from src.dataset import load_processed_split
from src.modeling.predict import log_experiment
from src.modeling.cnn import (
    set_seed, get_transforms, FASDataset, make_loader,
    build_model, count_parameters, train_cnn, evaluate_cnn,
)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
if device.type != 'cuda':
    print("AVISO: sem GPU. O teste roda na CPU (mais lento); nao treine sem GPU.")
print(f"Experimento: {EXPERIMENT}")

In [ ]:
# Carrega as imagens e labels pre-processadas com MTCNN (mesmo split dos classicos)

train_paths, y_train = load_processed_split(TRAIN_DIR)
val_paths, y_val = load_processed_split(VAL_DIR)

print(f"Train: {len(train_paths)} | Validation: {len(val_paths)}")

In [ ]:
# Cria os diretorios necessarios

RUNS_DIR = os.path.join(DRIVE_ROOT, 'experiments', 'cnn_densenet201', 'runs')
CKPT_DIR = os.path.join(DRIVE_ROOT, 'models', 'cnn_densenet201')
os.makedirs(RUNS_DIR, exist_ok=True)
os.makedirs(CKPT_DIR, exist_ok=True)
CACHE_DIR = os.path.join(DRIVE_ROOT, 'cache', 'cnn_160')

In [ ]:
# Carrega as imagens para a memoria e cria os dataloaders.
# Na primeira vez le do Drive e salva um unico arquivo de cache (.npz);
# nas proximas sessoes carrega esse arquivo, que e bem mais rapido.
# Augmentation so no treino; a validacao usa so normalizacao.

import time

t0 = time.time()
train_tf, eval_tf = get_transforms(ROTATION_DEGREES, BRIGHTNESS)

train_images = load_images(train_paths, os.path.join(CACHE_DIR, 'train.npz'))
val_images = load_images(val_paths, os.path.join(CACHE_DIR, 'val.npz'))

train_ds = FASDataset(train_images, y_train, train_tf)
val_ds = FASDataset(val_images, y_val, eval_tf)

val_loader = make_loader(val_ds, BATCH_SIZE, shuffle=False)
print(f"Dados prontos em {time.time() - t0:.1f}s")

In [ ]:
# Confere quantos parametros treinam neste experimento

m = build_model(EXPERIMENT, dropout=DROPOUT)
total, trainable = count_parameters(m)
print(f"{EXPERIMENT}: {trainable:,} de {total:,} parametros treinaveis "
      f"({100*trainable/total:.1f}%)")
del m

## Treino e validacao

Uma execucao por seed. O checkpoint com menor loss de validacao e salvo no
Drive. A validacao e usada so para escolher o checkpoint e comparar os
experimentos; o teste fica para a ultima secao.

In [ ]:
# Treina uma vez por seed, avalia na validacao e loga

import time

val_summary = []

for seed in SEEDS:
    exp_id = f"{EXPERIMENT}_seed{seed}"
    exp_dir = os.path.join(RUNS_DIR, exp_id)
    os.makedirs(exp_dir, exist_ok=True)
    ckpt_path = os.path.join(CKPT_DIR, f"{exp_id}.pt")

    print(f"\n===== {exp_id} =====")
    run_start = time.time()
    set_seed(seed)
    model = build_model(EXPERIMENT, dropout=DROPOUT)
    train_loader = make_loader(train_ds, BATCH_SIZE, shuffle=True, seed=seed)

    model, history = train_cnn(
        model, train_loader, val_loader, y_val, device,
        lr_head=LR_HEAD, lr_backbone=LR_BACKBONE,
        max_epochs=MAX_EPOCHS, patience=PATIENCE,
        checkpoint_path=ckpt_path,
    )
    history.to_csv(os.path.join(exp_dir, 'history.csv'), index=False)

    run_time = time.time() - run_start
    print(f"Tempo total da seed {seed}: {run_time/60:.1f} min")

    val_results = evaluate_cnn(model, val_loader, y_val, device)
    print(f"Validation -> HTER: {val_results['HTER']*100:.2f}% | "
          f"AUC: {val_results['AUC']:.4f} | EER: {val_results['EER']*100:.2f}%")

    log_experiment(
        exp_id=exp_id,
        metodo=METODO,
        feature_config=FEATURE_CONFIG,
        modelo_config=f"{MODELO_CONFIG}, seed={seed}",
        hter_val=val_results['HTER'],
        auc_val=val_results['AUC'],
        eer_val=val_results['EER'],
    )
    val_summary.append({'seed': seed, 'HTER': val_results['HTER'],
                        'AUC': val_results['AUC'], 'EER': val_results['EER'],
                        'epochs': len(history),
                        'time_min': run_time / 60})

    del model
    torch.cuda.empty_cache()

In [ ]:
# Resumo da validacao (media e desvio padrao entre seeds)

df_val = pd.DataFrame(val_summary)
print(df_val.to_string(index=False))
print(f"\nHTER val: {df_val.HTER.mean()*100:.2f}% +- {df_val.HTER.std()*100:.2f}")
print(f"AUC val:  {df_val.AUC.mean():.4f} +- {df_val.AUC.std():.4f}")
print(f"EER val:  {df_val.EER.mean()*100:.2f}% +- {df_val.EER.std()*100:.2f}")
print(f"Tempo medio por seed: {df_val.time_min.mean():.1f} min")

# Linha unica do experimento (media das seeds), no mesmo formato do HOG:
# exp_id = nome do experimento, metricas de validacao preenchidas.
log_experiment(
    exp_id=EXPERIMENT,
    metodo=METODO,
    feature_config=FEATURE_CONFIG,
    modelo_config=MODELO_CONFIG,
    hter_val=df_val.HTER.mean(),
    auc_val=df_val.AUC.mean(),
    eer_val=df_val.EER.mean(),
    obs=f"Media de {len(SEEDS)} seeds (desvio HTER {df_val.HTER.std()*100:.2f} p.p.)",
)

## Teste (rodar somente no final)

So rode esta secao depois de ter os 3 experimentos na validacao. Ela usa os
checkpoints salvos e avalia cada seed uma vez no conjunto de teste.
Mantenha `RUN_TEST = False` ate la.

In [ ]:
RUN_TEST = False

In [ ]:
if RUN_TEST:
    # Define de novo o que o teste precisa, para funcionar mesmo depois de
    # reiniciar o runtime (nao depende das celulas de treino).
    _, eval_tf = get_transforms(ROTATION_DEGREES, BRIGHTNESS)
    RUNS_DIR = os.path.join(DRIVE_ROOT, 'experiments', 'cnn_densenet201', 'runs')
    CKPT_DIR = os.path.join(DRIVE_ROOT, 'models', 'cnn_densenet201')
    CACHE_DIR = os.path.join(DRIVE_ROOT, 'cache', 'cnn_160')
    os.makedirs(RUNS_DIR, exist_ok=True)
    os.makedirs(CKPT_DIR, exist_ok=True)
    print(f"Experimento: {EXPERIMENT} | checkpoints: {sorted(f for f in os.listdir(CKPT_DIR) if f.startswith(EXPERIMENT))}")

    test_paths, y_test = load_processed_split(TEST_DIR)
    test_images = load_images(test_paths, os.path.join(CACHE_DIR, 'test.npz'))
    test_ds = FASDataset(test_images, y_test, eval_tf)
    test_loader = make_loader(test_ds, BATCH_SIZE, shuffle=False)
    print(f"Test: {len(test_paths)}")

    test_summary = []
    for seed in SEEDS:
        exp_id = f"{EXPERIMENT}_seed{seed}"
        exp_dir = os.path.join(RUNS_DIR, exp_id)

        model = build_model(EXPERIMENT, dropout=DROPOUT, pretrained=False)
        model.load_state_dict(torch.load(os.path.join(CKPT_DIR, f"{exp_id}.pt"), map_location=device))
        model.to(device)

        res = evaluate_cnn(model, test_loader, y_test, device, desc=f"teste {exp_id}")
        print(f"{exp_id} -> HTER: {res['HTER']*100:.2f}% | AUC: {res['AUC']:.4f} | "
              f"EER: {res['EER']*100:.2f}% | FAR: {res['FAR']*100:.2f}% | FRR: {res['FRR']*100:.2f}%")

        log_experiment(
            exp_id=exp_id + '_FINAL_TEST',
            metodo=METODO,
            feature_config=FEATURE_CONFIG,
            modelo_config=f"{MODELO_CONFIG}, seed={seed}",
            hter_test=res['HTER'],
            auc_test=res['AUC'],
            eer_test=res['EER'],
            obs='Test metric for one seed, evaluated once',
        )

        pd.DataFrame({
            'image_path': test_paths,
            'label_true': y_test,
            'label_pred': res['y_pred'],
            'score': res['y_scores'],
        }).to_csv(os.path.join(exp_dir, 'predictions_test.csv'), index=False)

        test_summary.append({'seed': seed, 'HTER': res['HTER'], 'AUC': res['AUC'],
                             'EER': res['EER'], 'FAR': res['FAR'], 'FRR': res['FRR']})
        del model
        torch.cuda.empty_cache()

    df_test = pd.DataFrame(test_summary)
    print(f"\nHTER teste: {df_test.HTER.mean()*100:.2f}% +- {df_test.HTER.std()*100:.2f}")
    print(f"AUC teste:  {df_test.AUC.mean():.4f} +- {df_test.AUC.std():.4f}")
    print(f"EER teste:  {df_test.EER.mean()*100:.2f}% +- {df_test.EER.std()*100:.2f}")
    print(f"FAR teste:  {df_test.FAR.mean()*100:.2f}% | FRR teste: {df_test.FRR.mean()*100:.2f}%")

    # Linha final reportada do experimento (media das seeds), no mesmo
    # formato do HOG: exp_id = <experimento>_FINAL_TEST.
    log_experiment(
        exp_id=EXPERIMENT + '_FINAL_TEST',
        metodo=METODO,
        feature_config=FEATURE_CONFIG,
        modelo_config=MODELO_CONFIG,
        hter_test=df_test.HTER.mean(),
        auc_test=df_test.AUC.mean(),
        eer_test=df_test.EER.mean(),
        obs=f"Final reported test metric for {METODO} {EXPERIMENT}: mean of {len(SEEDS)} seeds (HTER std {df_test.HTER.std()*100:.2f} p.p.)",
    )
else:
    print("RUN_TEST = False: teste nao executado.")